In [ ]:
import os
import time
import asyncio
import nest_asyncio
from openpyxl import load_workbook
from selenium_driverless import webdriver
from selenium_driverless.types.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import warnings
warnings.filterwarnings('ignore')

In [ ]:
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

In [ ]:
nest_asyncio.apply()
async def get_price_and_image_link(link, driver):
    await driver.get(link, wait_load=True, timeout=150)
    try:
        WebDriverWait(driver, 20).until(EC.visibility_of_element_located((By.XPATH, '//aside[@class="dialog fade js-lh-drawer lh-drawer in"]')))
        cookies = await driver.find_element(By.XPATH, '//aside[@class="dialog fade js-lh-drawer lh-drawer in"]')
        await driver.execute_script("arguments[0].click();", cookies)
    except:
        pass
    try:
        await WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '//*[@id="js-sirv-spin"]')))
        image_link_element = await driver.find_element(By.XPATH, '//meta[@property="og:image"]')
        image_link = await image_link_element.get_attribute("content")
        price_element = await driver.find_element(By.XPATH, '//*[@id="pageWrapper"]/div[3]/div/div[4]/div[3]/div/div[2]/div[1]/div/div/div[2]')
        price = await price_element.text
        return image_link, price
    except:
        return None, None
            


async def main():
    options = webdriver.ChromeOptions()
    async with webdriver.Chrome(options=options) as driver:
        await driver.maximize_window()
        excel_file_path = os.path.join(os.getcwd(), "LIVE Toolbox Inventory MASTER (1).xlsx")
        
        sheets = ["2", ]
        for sheet in sheets:
            try:
                workbook = load_workbook(excel_file_path)
                sheet = workbook[sheet]
                link_row = 3
                price_row = 3
                link_column = 1
                price_column = 5
                # Iterate over the Column C
                for row in sheet.iter_rows(min_row=2, max_col=3, values_only=True):
                    search_query = row[2]  # Column C
                    if search_query is None or search_query == "Part Number (GM)":
                        continue
                    else:
                        print(search_query)
                        link = "https://www.partstown.com/parts?q=" + str(search_query)
                        image_link, price = await get_price_and_image_link(link, driver)
                        if image_link is None or price is None:
                            link_row += 1
                            price_row += 1
                            continue
                        sheet.cell(row=link_row, column=link_column, value=image_link.strip())
                        sheet.cell(row=price_row, column=price_column, value=price.strip())
                        link_row += 1
                        price_row += 1
                        workbook.save(excel_file_path)
            except Exception as e:
                print(f"Error: {e}")

if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
# await WebDriverWait(driver, 10).until(EC.presence_of_elements_located((By.XPATH, f'//li[contains(text(), "Mfr Part #: ")]//following-sibling::strong[contains(text(), \'{link.split("q=")[1]}\')]//ancestor::div[@class="js-multi-product-item-plp"]//div[@class="name"]/a')))
#         listOfPros = await driver.find_element(By.XPATH, f'//li[contains(text(), "Mfr Part #: ")]//following-sibling::strong[contains(text(), \'{link.split("q=")[1]}\')]//ancestor::div[@class="js-multi-product-item-plp"]//div[@class="name"]/a')
#         print("Ya list nikli ha", listOfPros)
#         url = listOfPros.get_attribute('href')
#         print(url)
#         time.sleep(100)
#         for pro in listOfPros:
#             print("Ya pro nikla ha", pro)
#             url = pro.get_attribute('href')
#             print("Ya url nikla ha", url)
#             await driver.get(url)
#             await WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '//*[@id="js-sirv-spin"]')))
#             image_link_element = await driver.find_element(By.XPATH, '//meta[@property="og:image"]')
#             image_link = await image_link_element.get_attribute("content")
#             price_element = await driver.find_element(By.XPATH, '//*[@id="pageWrapper"]/div[3]/div/div[4]/div[3]/div/div[2]/div[1]/div/div/div[2]')
#             price = await price_element.text
#             if image_link or price:
#                 return image_link, price 